# Tutorial 19: Securing Your Travel Agent MCP Server

## 🎯 The Scenario: Building a Production-Ready AI Travel Agent

**You are a Solutions Architect at Contoso Travel**, a travel booking company building an AI-powered travel assistant. Your team has created a **Travel MCP Server** that provides real-time access to:

- ✈️ **Flight Search** - Query airline APIs for available flights
- 🏨 **Hotel Availability** - Check room inventory across hotel chains  
- 💱 **Currency Conversion** - Real-time exchange rates

The MCP server is deployed on **Azure Container Apps** and works great in development. But now the CISO asks:

> *"How do we secure this before production? We need to know who's calling our APIs, prevent abuse, and ensure only authorized applications can access our travel data."*

---

## 🏗️ Architecture Decision: Why APIM + Entra ID?

As the architect, you need to answer these questions:

| Security Concern | Solution | Why This Matters |
|-----------------|----------|------------------|
| **Who is calling?** | Entra ID JWT tokens | Audit trail per user/service |
| **Are they allowed?** | APIM subscription keys | API key management & revocation |
| **Rate limiting?** | APIM policies | Prevent abuse, ensure fair usage |
| **Single entry point?** | APIM gateway | Hide backend, centralize security |
| **Credential management?** | Project Connections | No secrets in code |

### The Complete Architecture

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                         CONTOSO TRAVEL AI PLATFORM                           │
├──────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│   ┌─────────────┐     ┌─────────────────┐     ┌─────────────────────────┐   │
│   │  End Users  │────>│  AI Foundry     │────>│  Azure API Management   │   │
│   │  (Travelers)│     │  Travel Agent   │     │  (Security Gateway)     │   │
│   └─────────────┘     └─────────────────┘     └───────────┬─────────────┘   │
│                              │                            │                  │
│                              │                            ▼                  │
│                    ┌─────────┴─────────┐     ┌─────────────────────────┐   │
│                    │ Project Connection │     │   Travel MCP Server     │   │
│                    │ (Stores API Key)   │     │  (Azure Container Apps) │   │
│                    └───────────────────┘     └─────────────────────────┘   │
│                                                          │                  │
│                                              ┌───────────┴───────────┐      │
│                                              ▼           ▼           ▼      │
│                                         [Flights]   [Hotels]   [Currency]   │
│                                                                              │
└──────────────────────────────────────────────────────────────────────────────┘
```

---

## 📋 What This Tutorial Covers

| Part | Topic | Architect's Question | Developer Task |
|------|-------|---------------------|----------------|
| **1** | Environment Setup | What config do we need? | Load credentials |
| **2** | Security Architecture | How do layers stack? | Understand the flow |
| **3** | MCP Test Helper | How do we validate? | Build test function |
| **4** | Direct MCP Test | Does backend work? | Baseline testing |
| **5** | Subscription Keys | How does key auth work? | Test APIM keys |
| **6** | JWT Validation | How do we add identity? | Configure APIM policy |
| **7** | Test JWT Flow | Does token auth work? | Validate with real tokens |
| **8** | Project Connections | How to store secrets? | Create secure connection |
| **9** | Auth Options | Which method when? | Choose auth strategy |
| **10** | Agent Testing | Does it all work? | End-to-end validation |
| **11** | APIM Policies | What else can we do? | Rate limits, logging |
| **12** | Summary | What did we achieve? | Document patterns |

---

## 🚀 Prerequisites

Before starting, ensure you have completed:

- **Tutorial 16**: Travel MCP Server deployed to Azure Container Apps
- **Tutorial 18**: APIM configured with your MCP Server as backend
- **Environment variables** configured in `.env` file

Let's build a production-ready secure AI travel agent! 🛫

---

## Part 1: Environment Setup

### 🏗️ Architect's Perspective

Before writing any code, the architect needs to identify **what configuration** the travel agent platform requires. This includes:

- **Identity**: Which Entra ID tenant owns our applications?
- **Gateway**: What's our APIM endpoint for the travel API?
- **Backend**: Where is the actual MCP server running?
- **Credentials**: How do we securely access APIM?

### 👨‍💻 Developer Task

Load these settings from environment variables. **Never hardcode secrets!** The `.env` file should contain:

```bash
# Identity (from Azure Portal → Entra ID)
AZURE_TENANT_ID=your-tenant-id

# APIM (from Tutorial 18)
APIM_GATEWAY_URL=https://apim-xxx.azure-api.net
APIM_SUBSCRIPTION_KEY=your-subscription-key

# Backend MCP Server (from Tutorial 16)
MCP_BACKEND_URL=https://travel-mcp.azurecontainerapps.io

# AI Foundry Project
AZURE_AI_PROJECT_ENDPOINT=https://xxx.services.ai.azure.com/api/projects/xxx
MCP_PROJECT_CONNECTION_ID=your-connection-id
```

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

# =============================================================================
# CONFIGURATION
# =============================================================================

# Azure AD / Entra ID
TENANT_ID = os.getenv("AZURE_TENANT_ID", "")

# APIM Configuration (from Tutorial 18)
APIM_GATEWAY_URL = os.getenv("APIM_GATEWAY_URL", "")  # e.g., https://apim-xxx.azure-api.net
APIM_SUBSCRIPTION_KEY = os.getenv("APIM_SUBSCRIPTION_KEY", "")
APIM_MCP_PATH = "/mcp/mcp"  # API path + operation

# Direct MCP Server URL (from Tutorial 16)
MCP_BACKEND_URL = os.getenv("MCP_BACKEND_URL", "")
MCP_DIRECT_URL = f"{MCP_BACKEND_URL}/mcp" if MCP_BACKEND_URL and not MCP_BACKEND_URL.endswith("/mcp") else MCP_BACKEND_URL

# Foundry Configuration
AZURE_AI_PROJECT_ENDPOINT = os.getenv("AZURE_AI_PROJECT_ENDPOINT", "")
MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")
MCP_PROJECT_CONNECTION_ID = os.getenv("MCP_PROJECT_CONNECTION_ID", "")  # Created in Foundry Portal

# Construct full APIM MCP URL
APIM_MCP_URL = f"{APIM_GATEWAY_URL}{APIM_MCP_PATH}" if APIM_GATEWAY_URL else ""

# Display configuration
print("\n" + "="*70)
print("CONFIGURATION")
print("="*70)
print(f"\n📍 Direct MCP URL: {MCP_DIRECT_URL}")
print(f"📍 APIM Gateway: {APIM_GATEWAY_URL}")
print(f"📍 APIM MCP URL: {APIM_MCP_URL}")
print(f"📍 Foundry Project: {AZURE_AI_PROJECT_ENDPOINT}")
print(f"📍 Tenant ID: {TENANT_ID[:20]}..." if TENANT_ID else "📍 Tenant ID: Not set")
print(f"📍 Subscription Key: {'✅ Set' if APIM_SUBSCRIPTION_KEY else '❌ Not set'}")
print(f"📍 Project Connection: {'✅ Set' if MCP_PROJECT_CONNECTION_ID else '❌ Not set'}")

---

## Part 2: Security Architecture

### 🏗️ Architect's Perspective

The CISO wants **defense in depth** - multiple security layers so that if one fails, others still protect us. Here's how we stack the security layers for our Travel MCP Server:

```
┌─────────────────────────────────────────────────────────────────────────┐
│ LAYER 1: Client Authentication                                          │
│ ────────────────────────────────────────────────────────────────────── │
│ WHO is making the request?                                              │
│ • Foundry Agent: Gets credentials from Project Connection               │
│ • Developer Testing: Manual headers in HTTP request                     │
│ • VS Code: Config in mcp.json                                          │
├─────────────────────────────────────────────────────────────────────────┤
│ LAYER 2: API Gateway (APIM Inbound Policies)                           │
│ ────────────────────────────────────────────────────────────────────── │
│ ARE THEY ALLOWED? And HOW MUCH can they use?                           │
│ • Subscription Key: "Do you have a valid API key?" (Ocp-Apim-...)     │
│ • JWT Validation: "Is this a real Entra ID user?" (validate-azure-ad) │
│ • Rate Limiting: "Are you making too many requests?" (rate-limit)     │
├─────────────────────────────────────────────────────────────────────────┤
│ LAYER 3: Backend Security                                               │
│ ────────────────────────────────────────────────────────────────────── │
│ PROTECT the actual service                                              │
│ • MCP Server on Container Apps (can be internal network only)          │
│ • APIM is the ONLY public entry point                                  │
│ • Backend never exposed directly to internet                           │
└─────────────────────────────────────────────────────────────────────────┘
```

### 🎯 Security Decision Matrix

When the architect reviews different MCP clients, each has different capabilities:

| Client Type | Can Send API Key? | Can Send JWT? | Best For |
|-------------|-------------------|---------------|----------|
| **Direct HTTP** | ✅ Yes | ✅ Yes | Testing, debugging |
| **Foundry Agent SDK** | ✅ Via Project Connection | ⚠️ Only via OAuth Passthrough | Production agents |
| **VS Code MCP** | ✅ Via mcp.json headers | ⚠️ Manual | Developer productivity |

### 💡 Key Insight for Developers

The Foundry Agent SDK has a **security restriction** - it blocks you from putting sensitive headers like `Authorization` directly in code. This is intentional! Instead, you must use **Project Connections** which:

1. Store secrets securely in Azure
2. Are injected at runtime by the SDK
3. Can be rotated without code changes
4. Are auditable by security teams

---

## Part 3: MCP Test Helper

### 🏗️ Architect's Perspective

Before deploying security policies, we need a way to **validate each layer independently**. The architect requires:

1. **Baseline tests**: Does the MCP server work at all?
2. **Negative tests**: Are unauthorized requests blocked?
3. **Positive tests**: Do valid credentials succeed?

### 👨‍💻 Developer Task

Build a reusable test function that:
- Sends the MCP `initialize` message (JSON-RPC 2.0 protocol)
- Accepts custom headers for testing different auth scenarios
- Returns clear success/failure with status codes

This helper will be used throughout the tutorial to validate each security layer.

In [ ]:
def test_mcp_endpoint(url: str, headers: dict = None, name: str = "Test"):
    """
    Test an MCP endpoint with the initialize message.
    Returns (success: bool, result: dict/str)
    """
    
    # MCP initialize request (JSON-RPC 2.0)
    mcp_initialize = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "initialize",
        "params": {
            "protocolVersion": "2024-11-05",
            "capabilities": {},
            "clientInfo": {"name": "test-client", "version": "1.0.0"}
        }
    }
    
    # Required headers for MCP Streamable HTTP
    default_headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream"
    }
    if headers:
        default_headers.update(headers)
    
    print(f"\n{'='*60}")
    print(f"🧪 {name}")
    print(f"{'='*60}")
    print(f"📍 URL: {url}")
    print(f"📤 Custom Headers: {list(headers.keys()) if headers else 'None'}")
    
    try:
        response = requests.post(url, json=mcp_initialize, headers=default_headers, timeout=30)
        print(f"📥 Status: {response.status_code}")
        
        if response.status_code == 200:
            content_type = response.headers.get('Content-Type', '')
            
            # Handle SSE response
            if 'text/event-stream' in content_type:
                for line in response.text.split('\n'):
                    if line.startswith('data:'):
                        data = line[5:].strip()
                        if data:
                            try:
                                result = json.loads(data)
                                if "result" in result:
                                    print(f"✅ SUCCESS - Server: {result['result'].get('serverInfo', {})}")
                                    return True, result
                            except json.JSONDecodeError:
                                continue
                return False, response.text
            else:
                # Handle JSON response
                try:
                    result = response.json()
                    if "result" in result:
                        print(f"✅ SUCCESS - Server: {result['result'].get('serverInfo', {})}")
                        return True, result
                except json.JSONDecodeError:
                    pass
                return False, response.text
        elif response.status_code == 401:
            print(f"🔒 UNAUTHORIZED - Authentication required")
            return False, "401 Unauthorized"
        elif response.status_code == 429:
            print(f"⏱️ RATE LIMITED - Too many requests")
            return False, "429 Too Many Requests"
        else:
            print(f"❌ FAILED - {response.status_code}: {response.text[:200]}")
            return False, response.text
    except Exception as e:
        print(f"❌ ERROR: {e}")
        return False, str(e)

---

## Part 4: Test Direct MCP Server (Baseline)

### 🏗️ Architect's Perspective

Before adding any security, we need to confirm the **Travel MCP Server itself works**. This is our baseline - if this fails, no amount of security configuration will help!

**Why test direct access?**
- Validates the MCP server is running and responding
- Confirms JSON-RPC protocol is working
- Establishes expected response format
- Helps isolate issues (backend vs. gateway)

### 👨‍💻 Developer Task

Hit the Container Apps URL directly (bypassing APIM). In production, this URL would be internal-only, but for testing we verify the backend is healthy.

**Expected Result**: ✅ 200 OK with MCP server info

In [ ]:
if MCP_DIRECT_URL:
    test_mcp_endpoint(MCP_DIRECT_URL, name="Direct MCP Server (No Auth)")
else:
    print("⚠️ MCP_BACKEND_URL not set - skipping direct test")

---

## Part 5: Test APIM Subscription Key Validation

### 🏗️ Architect's Perspective

**Subscription keys** are APIM's simplest security mechanism. Think of them as API keys that:

- **Identify** the application/team calling your API
- **Enable** usage tracking and billing per subscription
- **Allow** key rotation without changing backend code

For the Travel MCP Server, the architect might create different subscriptions:
- `travel-internal` - For internal applications (higher limits)
- `travel-partner` - For partner integrations (standard limits)  
- `travel-public` - For public API (lower limits, stricter quotas)

### 👨‍💻 Developer Task

Test three scenarios to validate subscription key enforcement:

| Test | Expected Result | What It Proves |
|------|-----------------|----------------|
| No key | 401 Unauthorized | APIM requires authentication |
| Invalid key | 401 Unauthorized | APIM validates keys |
| Valid key | 200 OK | Authorized access works |

**If all three pass, APIM subscription key security is working!**

In [ ]:
print("\n" + "="*70)
print("TESTING APIM SUBSCRIPTION KEY VALIDATION")
print("="*70)

if APIM_MCP_URL and APIM_SUBSCRIPTION_KEY:
    # Test 1: No subscription key (should fail)
    print("\n📌 Test 1: No subscription key (expect 401)")
    test_mcp_endpoint(APIM_MCP_URL, name="APIM - No Key")
    
    # Test 2: Invalid subscription key (should fail)
    print("\n📌 Test 2: Invalid subscription key (expect 401)")
    test_mcp_endpoint(
        APIM_MCP_URL, 
        headers={"Ocp-Apim-Subscription-Key": "invalid-key"}, 
        name="APIM - Invalid Key"
    )
    
    # Test 3: Valid subscription key (should succeed)
    print("\n📌 Test 3: Valid subscription key (expect 200)")
    test_mcp_endpoint(
        APIM_MCP_URL, 
        headers={"Ocp-Apim-Subscription-Key": APIM_SUBSCRIPTION_KEY}, 
        name="APIM - Valid Key"
    )
else:
    print("⚠️ APIM configuration incomplete - check APIM_GATEWAY_URL and APIM_SUBSCRIPTION_KEY")

---

## Part 6: APIM JWT Validation Policy

### 🏗️ Architect's Perspective

Subscription keys tell us **which application** is calling, but not **which user**. The CISO asks:

> *"When the travel agent books a $10,000 flight, I need to know exactly which employee authorized it. Can we trace every action to a specific person?"*

**Solution**: Add Entra ID JWT validation on top of subscription keys.

| Security Layer | What It Answers | Example |
|----------------|-----------------|---------|
| Subscription Key | Which app? | "Travel Web App" |
| JWT Token | Which user? | "alice@contoso.com" |
| Combined | Full audit trail | "Alice from Travel Web App searched NYC→LAX" |

### Benefits for Different Stakeholders

| Role | Benefit |
|------|---------|
| **CISO** | Complete audit trail, compliance reporting |
| **Architect** | Role-based access control possible |
| **Developer** | User context available in logs |
| **Operations** | Easier troubleshooting ("Who did this?") |

### 👨‍💻 Developer Task

Apply the JWT validation policy in Azure Portal:

1. **Portal Path**: APIM → APIs → travel-mcp → Design → Inbound policies
2. **Click**: `</>` to edit XML
3. **Paste**: The policy generated below
4. **Save**: Test will change from key-only to key+JWT required

⚠️ **Breaking Change**: Once enabled, requests with ONLY a subscription key will be **rejected**!

In [ ]:
# Generate policy with your tenant ID
if TENANT_ID:
    print("\n" + "="*70)
    print("APIM JWT VALIDATION POLICY")
    print("="*70)
    print(f"""
Copy this policy to: Azure Portal → APIM → APIs → travel-mcp → Inbound policies

<policies>
    <inbound>
        <base />
        <validate-azure-ad-token 
            tenant-id="{TENANT_ID}" 
            header-name="Authorization" 
            failed-validation-httpcode="401" 
            failed-validation-error-message="Unauthorized">
            <audiences>
                <audience>https://management.azure.com</audience>
            </audiences>
        </validate-azure-ad-token>
    </inbound>
    <backend><base /></backend>
    <outbound><base /></outbound>
    <on-error><base /></on-error>
</policies>
""")
else:
    print("⚠️ AZURE_TENANT_ID not set")

---

## Part 7: Test JWT Validation

### 🏗️ Architect's Perspective

Now we validate that our JWT policy is actually enforcing authentication. We'll run three tests:

| Test | What We Send | Expected Result | What It Proves |
|------|--------------|-----------------|----------------|
| Key only | Subscription key, no JWT | ❌ 401 | JWT is required |
| Fake JWT | Key + made-up token | ❌ 401 | APIM validates signature |
| Real JWT | Key + Entra ID token | ✅ 200 | Authorized user can access |

### 💡 How JWT Validation Works

When APIM receives a request with the JWT policy enabled:

```
1. Extract token from Authorization header
2. Decode the JWT (header.payload.signature)
3. Verify signature using Entra ID's public keys
4. Check claims: tenant ID, audience, expiration
5. If all pass → forward to backend
6. If any fail → return 401 Unauthorized
```

### 👨‍💻 Developer Task

First, we'll get a real Entra ID token using `DefaultAzureCredential`. This uses your logged-in Azure identity (from `az login` or VS Code Azure extension).

In [ ]:
from azure.identity import DefaultAzureCredential
import base64

def get_entra_token():
    """Get a real Entra ID token using DefaultAzureCredential."""
    try:
        credential = DefaultAzureCredential()
        token = credential.get_token("https://management.azure.com/.default")
        return token.token
    except Exception as e:
        print(f"❌ Failed to get token: {e}")
        return None

# Get token
print("\n" + "="*70)
print("GETTING ENTRA ID TOKEN")
print("="*70)

jwt_token = get_entra_token()
if jwt_token:
    print(f"✅ Got JWT token (length: {len(jwt_token)} chars)")
    
    # Show some claims
    try:
        payload = jwt_token.split('.')[1]
        padding = 4 - len(payload) % 4
        if padding != 4:
            payload += '=' * padding
        decoded = json.loads(base64.b64decode(payload))
        
        print(f"\n📋 Token Claims:")
        for key in ['name', 'upn', 'oid', 'tid']:
            if key in decoded:
                value = decoded[key]
                if key in ['oid', 'tid']:
                    value = value[:8] + "..."
                print(f"   • {key}: {value}")
    except:
        pass
else:
    print("⚠️ Could not get JWT token")

In [ ]:
# Test JWT Validation - Expected Failures
print("\n" + "="*70)
print("TESTING JWT POLICY ENFORCEMENT")
print("="*70)
print("\n⚠️  These tests assume JWT policy is ENABLED in APIM")

if APIM_MCP_URL and APIM_SUBSCRIPTION_KEY:
    
    # Test 1: Subscription Key ONLY (should fail with JWT policy)
    print("\n📌 Test 1: Subscription Key Only (expect 401 if JWT policy enabled)")
    success1, _ = test_mcp_endpoint(
        APIM_MCP_URL, 
        headers={"Ocp-Apim-Subscription-Key": APIM_SUBSCRIPTION_KEY}, 
        name="Key Only - No JWT"
    )
    
    # Test 2: Fake JWT Token
    print("\n📌 Test 2: Fake JWT Token (expect 401)")
    FAKE_JWT = "eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJzdWIiOiJmYWtlIn0.fake"
    success2, _ = test_mcp_endpoint(
        APIM_MCP_URL, 
        headers={
            "Ocp-Apim-Subscription-Key": APIM_SUBSCRIPTION_KEY,
            "Authorization": f"Bearer {FAKE_JWT}"
        }, 
        name="Fake JWT + Valid Key"
    )
    
    # Summary
    if not success1 and not success2:
        print("\n✅ JWT POLICY IS WORKING!")
        print("   Both unauthorized requests were rejected.")
    elif success1:
        print("\n⚠️  JWT policy may not be enabled - key-only request succeeded")

In [ ]:
# Test with REAL JWT Token - Should Succeed!
print("\n" + "="*70)
print("TESTING WITH REAL ENTRA ID TOKEN")
print("="*70)

if APIM_MCP_URL and APIM_SUBSCRIPTION_KEY and jwt_token:
    
    print("\n📌 Test: Real JWT Token + Valid Subscription Key (expect 200)")
    success, result = test_mcp_endpoint(
        APIM_MCP_URL, 
        headers={
            "Ocp-Apim-Subscription-Key": APIM_SUBSCRIPTION_KEY,
            "Authorization": f"Bearer {jwt_token}"
        }, 
        name="Real JWT + Valid Key"
    )
    
    if success:
        print("\n🎉 SUCCESS! Complete security flow works:")
        print("   ✅ Subscription key validated")
        print("   ✅ JWT token validated")
        print("   ✅ Request forwarded to MCP server")
else:
    print("⚠️ Missing configuration or JWT token")

---

## Part 8: Project Connections for Foundry Agents

### 🏗️ Architect's Perspective

The development team asks: *"Great, JWT + subscription key works in our tests. But how does the AI agent get these credentials?"*

**The Problem**: 
- Agents run in Azure AI Foundry, not on developer laptops
- We can't hardcode secrets in agent code (security risk!)
- Agents need credentials at runtime to call APIM

**The Solution**: **Project Connections** - secure credential storage in Azure AI Foundry

```
┌─────────────────────────────────────────────────────────────────────────┐
│  WITHOUT Project Connections (Bad ❌)                                   │
│  ─────────────────────────────────────────────────                     │
│  headers = {"Ocp-Apim-Subscription-Key": "abc123"}  # Secret in code!  │
│                                                                         │
├─────────────────────────────────────────────────────────────────────────┤
│  WITH Project Connections (Good ✅)                                     │
│  ─────────────────────────────────────────────────                     │
│  MCPTool(                                                               │
│      server_url=APIM_URL,                                               │
│      project_connection_id="connections/apim-mcp-connection"            │
│  )                                                                      │
│  # SDK injects the key at runtime from secure storage                   │
└─────────────────────────────────────────────────────────────────────────┘
```

### Benefits for the Team

| Role | Benefit |
|------|---------|
| **Security** | Secrets never in code, auditable access |
| **DevOps** | Rotate keys without code deployments |
| **Developer** | Simple API, no secret management |
| **Compliance** | Centralized credential governance |

### 👨‍💻 Developer Task: Create a Project Connection

1. Open **Azure AI Foundry Portal**: https://ai.azure.com
2. Go to: **Your Project → Management → Connected resources**
3. Click: **+ New connection → Custom keys**
4. Configure:
   - **Name**: `apim-mcp-connection`
   - **Key name**: `Ocp-Apim-Subscription-Key`
   - **Key value**: Your APIM subscription key
5. **Save** and copy the Connection ID to `.env`

In [ ]:
# List existing connections
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

print("\n" + "="*70)
print("PROJECT CONNECTIONS")
print("="*70)

if AZURE_AI_PROJECT_ENDPOINT:
    try:
        credential = DefaultAzureCredential()
        project_client = AIProjectClient(
            credential=credential,
            endpoint=AZURE_AI_PROJECT_ENDPOINT
        )
        
        connections = list(project_client.connections.list())
        print(f"\n📋 Found {len(connections)} connections:")
        
        for conn in connections:
            conn_type = getattr(conn, 'connection_type', 'Unknown')
            conn_name = getattr(conn, 'name', 'Unknown')
            conn_id = getattr(conn, 'id', '')
            
            marker = "  " if conn_type != 'Custom' else "👉"
            print(f"   {marker} {conn_name} (Type: {conn_type})")
            
            if conn_type == 'Custom':
                print(f"      ID: {conn_id}")
        
        if MCP_PROJECT_CONNECTION_ID:
            print(f"\n✅ MCP_PROJECT_CONNECTION_ID is set: {MCP_PROJECT_CONNECTION_ID.split('/')[-1]}")
        else:
            print("\n⚠️ MCP_PROJECT_CONNECTION_ID not set - create a Custom connection and set it")
            
    except Exception as e:
        print(f"❌ Error: {e}")
else:
    print("⚠️ AZURE_AI_PROJECT_ENDPOINT not set")

---

## Part 9: MCP Authentication Options

### 🏗️ Architect's Decision Guide

The architect needs to choose the right authentication method for each scenario:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│  DECISION TREE: Which MCP Auth to Use?                                      │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                              │
│  Is this development/testing?                                                │
│      YES → Option 1: Direct URL (no auth)                                   │
│      NO  ↓                                                                  │
│                                                                              │
│  Do you need user-level audit trails?                                        │
│      NO  → Option 2: Key-based (APIM + Subscription Key)                    │
│      YES ↓                                                                  │
│                                                                              │
│  Is APIM JWT policy enabled?                                                 │
│      NO  → Option 2: Key-based (sufficient for app-level identity)          │
│      YES → Option 3: OAuth Identity Passthrough (user JWT flows through)    │
│                                                                              │
└─────────────────────────────────────────────────────────────────────────────┘
```

### Authentication Options Summary

| Option | Security Level | Setup Complexity | Use Case |
|--------|---------------|------------------|----------|
| **1. Direct** | 🔓 None | Low | Dev/testing, internal networks |
| **2. Key-based** | 🔐 App-level | Medium | Production with app identity |
| **3. OAuth** | 🔒 User-level | High | Full audit trails, RBAC |

### ⚠️ SDK Security Restriction

The Foundry SDK **intentionally blocks** sensitive headers in code:

```python
# This will FAIL with security error:
MCPTool(
    server_url=URL,
    headers={"Authorization": "Bearer xxx"}  # ❌ Blocked!
)

# This is the CORRECT approach:
MCPTool(
    server_url=URL,
    project_connection_id="connections/my-connection"  # ✅ Secure!
)
```

This design forces developers to use secure credential storage rather than embedding secrets in code.

### 👨‍💻 Developer Task

Create MCP tools for each available authentication option and select the best one based on your configuration.

In [ ]:
from azure.ai.projects.models import MCPTool, PromptAgentDefinition

print("\n" + "="*70)
print("MCP AUTHENTICATION OPTIONS")
print("="*70)

# Option 1: Direct URL (no auth)
print("\n📌 Option 1: Direct URL (no auth)")
if MCP_DIRECT_URL:
    mcp_tool_direct = MCPTool(
        server_label="travel-mcp-direct",
        server_url=MCP_DIRECT_URL
    )
    print(f"   ✅ Created: {mcp_tool_direct.server_url}")
else:
    mcp_tool_direct = None
    print("   ❌ Skipped: MCP_BACKEND_URL not set")

# Option 2: APIM + Subscription Key
print("\n📌 Option 2: APIM + Subscription Key (via Project Connection)")
if APIM_MCP_URL and MCP_PROJECT_CONNECTION_ID:
    mcp_tool_apim = MCPTool(
        server_label="travel-mcp-apim",
        server_url=APIM_MCP_URL,
        project_connection_id=MCP_PROJECT_CONNECTION_ID
    )
    print(f"   ✅ Created: {mcp_tool_apim.server_url}")
    print(f"   Connection: {MCP_PROJECT_CONNECTION_ID.split('/')[-1]}")
else:
    mcp_tool_apim = None
    print("   ❌ Skipped: APIM_MCP_URL or MCP_PROJECT_CONNECTION_ID not set")

# Option 3: OAuth Identity Passthrough (requires Portal setup)
print("\n📌 Option 3: OAuth Identity Passthrough")
OAUTH_CONNECTION_ID = os.getenv("MCP_OAUTH_CONNECTION_ID", "")
if OAUTH_CONNECTION_ID:
    mcp_tool_oauth = MCPTool(
        server_label="travel-mcp-oauth",
        server_url=APIM_MCP_URL,
        project_connection_id=OAUTH_CONNECTION_ID
    )
    print(f"   ✅ Created: OAuth passthrough")
else:
    mcp_tool_oauth = None
    print("   ⚠️  Not configured (requires Foundry Portal setup)")

# Select best available option
if mcp_tool_oauth:
    mcp_tool = mcp_tool_oauth
    auth_mode = "OAuth Identity Passthrough"
elif mcp_tool_apim:
    mcp_tool = mcp_tool_apim
    auth_mode = "APIM + Subscription Key"
elif mcp_tool_direct:
    mcp_tool = mcp_tool_direct
    auth_mode = "Direct (no auth)"
else:
    mcp_tool = None
    auth_mode = None

print(f"\n🎯 Selected: {auth_mode or 'None available'}")

---

## Part 10: Create and Test the Travel Agent

### 🏗️ Architect's Perspective

This is the **moment of truth** - we bring together all the pieces:

```
User Query: "Find me flights from New York to Los Angeles"
     │
     ▼
┌─────────────────────────────────────────────────────────────────┐
│  TRAVEL ASSISTANT AGENT (Azure AI Foundry)                      │
│  • Receives natural language query                              │
│  • Identifies need for flight search tool                       │
│  • Calls MCP Server via configured authentication               │
└─────────────────────────────────────────────────────────────────┘
     │
     ▼
┌─────────────────────────────────────────────────────────────────┐
│  AZURE API MANAGEMENT                                           │
│  • Validates subscription key (from Project Connection)         │
│  • Validates JWT token (if OAuth configured)                    │
│  • Applies rate limiting                                        │
│  • Logs request for audit                                       │
└─────────────────────────────────────────────────────────────────┘
     │
     ▼
┌─────────────────────────────────────────────────────────────────┐
│  TRAVEL MCP SERVER (Azure Container Apps)                       │
│  • Executes search_flights tool                                 │
│  • Queries flight APIs                                          │
│  • Returns structured flight data                               │
└─────────────────────────────────────────────────────────────────┘
     │
     ▼
Agent synthesizes response: "I found 5 flights. The cheapest is..."
```

### 👨‍💻 Developer Task

1. Create the Travel Assistant agent with the selected MCP tool
2. Build a helper function to handle MCP tool approvals
3. Test with real travel queries (currency, flights, hotels)

In [ ]:
# Create agent
import time

print("\n" + "="*70)
print("CREATING TRAVEL ASSISTANT AGENT")
print("="*70)

if mcp_tool and AZURE_AI_PROJECT_ENDPOINT:
    try:
        openai_client = project_client.get_openai_client()
        
        agent_definition = PromptAgentDefinition(
            model=MODEL_DEPLOYMENT,
            instructions="""You are a helpful travel assistant that can:
- Search for flights between cities
- Check hotel availability
- Convert currencies

Always use the available MCP tools to provide accurate information.""",
            tools=[mcp_tool]
        )
        
        agent = project_client.agents.create(
            name=f"travel-assistant-{int(time.time())}",
            definition=agent_definition
        )
        
        print(f"\n✅ Agent created!")
        print(f"   Name: {agent.name}")
        print(f"   Model: {MODEL_DEPLOYMENT}")
        print(f"   Auth: {auth_mode}")
        
    except Exception as e:
        print(f"❌ Error creating agent: {e}")
else:
    print("⚠️ Cannot create agent - missing MCP tool or project endpoint")

In [ ]:
def run_agent_query(query: str, tool=None):
    """
    Run a query with the travel agent.
    Handles MCP approval requests and OAuth consent.
    """
    
    tool = tool or mcp_tool
    
    print(f"\n{'='*60}")
    print(f"🤖 Query: {query}")
    print(f"📍 Server: {tool.server_url}")
    print(f"{'='*60}")
    
    try:
        response = openai_client.responses.create(
            model=MODEL_DEPLOYMENT,
            input=query,
            tools=[tool]
        )
        
        # Handle MCP approval requests
        max_iterations = 5
        for _ in range(max_iterations):
            approval_requests = []
            
            for item in response.output:
                item_type = getattr(item, 'type', 'unknown')
                
                if item_type == 'oauth_consent_request':
                    consent_link = getattr(item, 'consent_link', None)
                    print(f"\n🔐 OAuth Sign-in Required!")
                    print(f"   Sign in at: {consent_link}")
                    return None
                
                if item_type == 'mcp_approval_request':
                    tool_name = getattr(item, 'name', 'unknown')
                    item_id = getattr(item, 'id', 'unknown')
                    print(f"\n🔧 Approving: {tool_name}")
                    approval_requests.append({
                        "type": "mcp_approval_response",
                        "approval_request_id": item_id,
                        "approve": True
                    })
                elif item_type == 'mcp_call':
                    tool_name = getattr(item, 'name', 'unknown')
                    print(f"✅ Executed: {tool_name}")
            
            if approval_requests:
                response = openai_client.responses.create(
                    model=MODEL_DEPLOYMENT,
                    previous_response_id=response.id,
                    input=approval_requests,
                    tools=[tool]
                )
            else:
                break
        
        # Print response
        print(f"\n📝 Response:")
        if hasattr(response, 'output_text') and response.output_text:
            print(response.output_text)
        else:
            for item in response.output:
                if hasattr(item, 'content'):
                    print(item.content)
        
        return response
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return None

In [ ]:
# Test the agent
if mcp_tool_direct:
    print("Testing with Direct URL (bypasses APIM):")
    run_agent_query("Convert 500 USD to EUR", mcp_tool_direct)

In [ ]:
# Test flight search
if mcp_tool_direct:
    run_agent_query("Search for flights from New York to Los Angeles for tomorrow", mcp_tool_direct)

---

## Part 11: Additional APIM Security Policies

### 🏗️ Architect's Perspective

The CISO comes back: *"JWT and subscription keys are great, but what about abuse prevention? What if someone gets a valid key and hammers our API?"*

APIM provides **policy-based protection** that the architect can layer on top of authentication:

| Policy | Problem It Solves | Business Impact |
|--------|------------------|-----------------|
| **Rate Limiting** | Too many requests per second | Protects backend from overload |
| **Quota** | Too much usage per month | Enforces subscription tiers |
| **IP Filtering** | Unauthorized networks | Corporate network only access |
| **Caching** | Repeated identical requests | Reduces cost, improves latency |
| **Logging** | "What happened?" | Debugging, compliance, audit |

### Policy Execution Order

Understanding the order helps architects design effective policies:

```
Request arrives at APIM
     │
     ▼ INBOUND POLICIES (in order)
     ├── 1. rate-limit      → Block if too many requests
     ├── 2. validate-jwt    → Block if no/invalid token
     ├── 3. ip-filter       → Block if wrong network
     ├── 4. quota           → Block if quota exceeded
     └── 5. trace           → Log the request
     │
     ▼ BACKEND
     └── Forward to MCP Server
     │
     ▼ OUTBOUND POLICIES
     └── Add response headers, logging
```

### 👨‍💻 Developer Task

Let's see rate limiting in action - this is the most commonly used protection policy.

In [ ]:
# Rate Limiting Policy
print("="*70)
print("RATE LIMITING POLICY")
print("="*70)
print("""
<!-- Add to <inbound> section -->
<rate-limit calls="2" renewal-period="60" 
    remaining-calls-header-name="X-RateLimit-Remaining"
    retry-after-header-name="Retry-After" />

How it works:
• Limits to 2 calls per 60 seconds (set low for testing!)
• Returns 429 Too Many Requests when exceeded
• Adds X-RateLimit-Remaining header
• In production, use higher values (e.g., 100/minute)
""")

In [ ]:
# Test Rate Limiting
def test_rate_limiting(url: str, num_requests: int = 5):
    """Send rapid requests to test rate limiting.
    
    With rate-limit set to 2 calls/60 seconds:
    - Requests 1-2: Should succeed (200)
    - Requests 3+: Should be rate limited (429)
    """
    
    print(f"🚀 Sending {num_requests} rapid requests (limit: 2/minute)...")
    
    # Get fresh JWT token
    credential = DefaultAzureCredential()
    token = credential.get_token("https://management.azure.com/.default").token
    
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
        "Authorization": f"Bearer {token}",
        "Ocp-Apim-Subscription-Key": APIM_SUBSCRIPTION_KEY
    }
    
    mcp_msg = {"jsonrpc": "2.0", "method": "initialize", "params": {
        "protocolVersion": "2024-11-05", 
        "clientInfo": {"name": "test", "version": "1.0"}
    }, "id": 1}
    
    for i in range(num_requests):
        try:
            response = requests.post(url, json=mcp_msg, headers=headers, timeout=10)
            remaining = response.headers.get("X-RateLimit-Remaining", "N/A")
            status = "✅" if response.status_code == 200 else f"🚫 {response.status_code}"
            print(f"   Request {i+1}: {status} | Remaining: {remaining}")
        except Exception as e:
            print(f"   Request {i+1}: ❌ Error")

if APIM_MCP_URL and APIM_SUBSCRIPTION_KEY:
    test_rate_limiting(APIM_MCP_URL, 5)
else:
    print("⚠️ APIM not configured")

In [ ]:
# Complete Production-Ready Policy
print("="*70)
print("PRODUCTION-READY MCP POLICY")
print("="*70)
print(f"""
<policies>
    <inbound>
        <base />
        
        <!-- JWT Authentication -->
        <validate-azure-ad-token tenant-id="{TENANT_ID or 'YOUR-TENANT-ID'}">
            <audiences>
                <audience>https://management.azure.com</audience>
            </audiences>
        </validate-azure-ad-token>
        
        <!-- Rate Limiting (adjust for your needs) -->
        <!-- Testing: calls="2" | Production: calls="100" -->
        <rate-limit calls="100" renewal-period="60" 
            remaining-calls-header-name="X-RateLimit-Remaining"
            retry-after-header-name="Retry-After" />
        
        <!-- Request Tracing -->
        <trace source="MCP-Gateway" severity="information">
            <message>@($"MCP: {{context.Request.Method}} from {{context.Request.IpAddress}}")</message>
        </trace>
    </inbound>
    
    <backend><base /></backend>
    <outbound><base /></outbound>
    <on-error><base /></on-error>
</policies>
""")

---

## Part 12: Summary - The Complete Security Story

### 🎯 What Contoso Travel Achieved

Remember where we started? The CISO asked: *"How do we secure this for production?"*

Here's the security architecture we built together:

```
┌─────────────────────────────────────────────────────────────────────┐
│                    CONTOSO TRAVEL - SECURED                         │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   👤 Customer                                                       │
│      │                                                             │
│      ▼                                                             │
│   🤖 AI Travel Agent (Azure AI Foundry)                            │
│      │                                                             │
│      │ MCPTool with Project Connection                             │
│      │ (credentials auto-managed)                                  │
│      │                                                             │
│      ▼                                                             │
│   🛡️ APIM Gateway ─────────────────────────────────────────────────│
│      │                                                             │
│      ├── ✓ Rate Limiting (2/minute for testing, 100/min prod)      │
│      ├── ✓ JWT Validation (Entra ID tokens)                        │
│      ├── ✓ Subscription Keys (per-consumer tracking)               │
│      ├── ✓ IP Filtering (optional corporate network only)          │
│      └── ✓ Logging to Application Insights                         │
│      │                                                             │
│      ▼                                                             │
│   ✈️ Travel MCP Server (Container Apps)                            │
│      │                                                             │
│      ├── /mcp/mcp - MCP Protocol Endpoint                          │
│      └── Tools: search_flights, check_hotel, convert_currency       │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

### 📋 Security Checklist

| Requirement | Solution | Status |
|-------------|----------|--------|
| **Authentication** | Entra ID JWT tokens | ✅ |
| **Authorization** | Subscription keys per consumer | ✅ |
| **Rate Limiting** | APIM policies (5/sec, 100/min) | ✅ |
| **Credential Management** | AI Foundry Project Connections | ✅ |
| **Audit Trail** | APIM + Application Insights | ✅ |
| **Network Security** | IP filtering (optional) | ⚙️ |
| **Zero Secrets in Code** | Project Connections | ✅ |

### 🏗️ Architect's Key Decisions

1. **APIM as Security Gateway** - Single point of enforcement
2. **Entra ID for Identity** - Corporate SSO, no custom auth
3. **Project Connections** - Developers never see credentials
4. **Policy-based Protection** - Declarative, auditable, changeable

### 👨‍💻 Developer Experience

What developers can now say:

> *"I just call `MCPTool(server_label='travel-mcp')` and everything works. 
> Security team handles the policies. I focus on building great travel experiences."*

### 🔗 What's Next?

Now that you have a secured MCP server, explore:
- **Tutorial 20**: Design patterns for multi-agent systems
- **Azure Monitor**: Set up alerts when rate limits are hit
- **Multiple Environments**: Use different subscription keys for dev/test/prod

---

**Congratulations!** You've built an enterprise-grade, secured AI agent with MCP tools. 🎉

In [ ]:
# Final Summary
print("\n" + "="*70)
print("TUTORIAL 19: FINAL STATUS")
print("="*70)

print("\n✅ WORKING:")
print("   • Direct MCP access (no auth)")
print("   • APIM subscription key validation")
print("   • APIM JWT token validation")
print("   • Foundry Agent with Project Connection")
print("   • Rate limiting policy")

print("\n❌ LIMITATIONS:")
print("   • Foundry Agent + APIM JWT policy (SDK can't inject JWT)")
print("   • OAuth Passthrough (requires Portal configuration)")

print("\n📚 NEXT TUTORIALS:")
print("   • Tutorial 16: Deploy MCP Server to Container Apps")
print("   • Tutorial 18: APIM Integration with MCP")
print("   • Tutorial 20: Advanced Orchestration Patterns")